In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("../data/raw/covid19br.csv")

print(df[df['viral'] == 1]['sharings'].describe())
print(df[df['viral'] == 0]['sharings'].describe())

df['text_count'] = df.groupby('text')['text'].transform('count')
print(df[['sharings', 'text_count']].corr())
print(df[df['words'] >= 15][['sharings', 'text_count', 'viral']].head(10))

count    14729.000000
mean        11.458144
std         15.551749
min          2.000000
25%          2.000000
50%          5.000000
75%         19.000000
max        123.000000
Name: sharings, dtype: float64
count    213539.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: sharings, dtype: float64
            sharings  text_count
sharings    1.000000   -0.100226
text_count -0.100226    1.000000
     sharings  text_count  viral
33          8           8      1
85          1           1      0
86         12          12      1
88         12          12      1
89         15          15      1
109         1           1      0
122         4           4      1
123         1           1      0
126         1           1      0
128         1           1      0


In [3]:
import re

# 1. drop hidden-media rows and short chat messages
d = df[(df['midia'] == 0) & (df['text'] != "<Arquivo de mídia oculto>")]
d = d[d['words'] >= 10].copy()
print("after filter:", len(d))

# 2. normalize text
url_re = re.compile(r"https?://\S+|www\.\S+")
def clean(t):
    t = url_re.sub(" ", t)
    t = re.sub(r"[^\w\s.,;:!?()%$/-]", " ", t)  # removes emojis, stars, ASCII art
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

d['clean'] = d['text'].apply(clean)
d = d[d['clean'].str.split().str.len() >= 10]
d['date_dt'] = pd.to_datetime(d['date'], format='%d/%m/%y')

# 3. collapse exact duplicates but keep spread info
agg = (d.groupby('clean')
         .agg(text=('text', 'first'),
              n_occurrences=('id', 'size'),
              n_groups=('group', 'nunique'),
              first_date=('date_dt', 'min'),
              max_sharings=('sharings', 'max'),
              viral=('viral', 'max'))
         .reset_index())

# 4. flag COVID-related messages
kw = r"covid|corona|quarentena|pandemia|cloroquina|vacina|isolamento"
agg['is_covid'] = agg['clean'].str.contains(kw, regex=True)

print("unique cleaned messages:", len(agg))
print("covid-related:", agg['is_covid'].sum())
print(agg[['n_occurrences', 'n_groups', 'max_sharings']].describe())

agg.to_csv("../data/processed/messages_clean.csv", index=False)

after filter: 33064
unique cleaned messages: 20176
covid-related: 2043
       n_occurrences      n_groups  max_sharings
count   20176.000000  20176.000000  20176.000000
mean        1.606215      1.303975      1.470212
std        16.036681      1.873320      2.546967
min         1.000000      1.000000      1.000000
25%         1.000000      1.000000      1.000000
50%         1.000000      1.000000      1.000000
75%         1.000000      1.000000      1.000000
max      2248.000000     63.000000    123.000000
